In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download("ai12-level1-project")

print("Path to competition files:", path)

In [ ]:
import getpass
import json
import shutil
from collections import defaultdict
from pathlib import Path

import torch
import yaml
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [ ]:
SRC = Path.home() / path / "sprint_ai_project1_data"
DST = Path("data/pill_yolo")

SRC_TRAIN_IMAGES = SRC / "train_images"
SRC_TEST_IMAGES = SRC / "test_images"
SRC_ANNOTATIONS = SRC / "train_annotations"

SUPER_KEY = ord("*")
USERNAME = getpass.getuser() or "team03"
torch.manual_seed(SUPER_KEY)
g = torch.Generator().manual_seed(SUPER_KEY)

test_size = 0.2

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
# fid_device = torch.device("cpu")

### 전체 category_id 수집 -> 0부터 연속 정수로 매핑

In [ ]:
cat_map = {}  # {원본 category_id: yolo_class_id}
cat_names = {}  # {yolo_class_id: 약품명}

all_jsons = sorted(SRC_ANNOTATIONS.rglob("*.json"))

print("jsons len", len(all_jsons))
for json_file in all_jsons:
    data = json.load(open(json_file))
    for cat in data["categories"]:
        if cat["id"] not in cat_map:
            idx = len(cat_map)
            cat_map[cat["id"]] = idx
            cat_names[idx] = cat["name"]

print(f"클래스 수: {len(cat_map)}")


### 이미지별로 annotation 모으기

In [ ]:
# key: 파일명(str), value: list of (class_id, bbox, img_w, img_h)
img_annotations = defaultdict(list)

for json_file in all_jsons:
    data = json.load(open(json_file))
    img_info = data["images"][0]
    file_name = img_info["file_name"]
    img_w, img_h = img_info["width"], img_info["height"]

    for ann in data["annotations"]:
        original_cat_id = ann["category_id"]
        yolo_cls = cat_map[original_cat_id]
        x_min, y_min, w, h = ann["bbox"]

        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        img_annotations[file_name].append((
            yolo_cls,
            x_center,
            y_center,
            norm_w,
            norm_h,
        ))

print(f"어노테이션 있는 이미지 수: {len(img_annotations)}")

### train val 8:2

In [ ]:
image_names = sorted(img_annotations.keys())
train_names, val_names = train_test_split(
    image_names, test_size=test_size, random_state=SUPER_KEY
)
print(f"train: {len(train_names)}, val: {len(val_names)}")

### YOLO COCO용 DIR 생성 + 파일 복사

In [ ]:
for split, names in [("train", train_names), ("val", val_names)]:
    image_dir = DST / "images" / split
    label_dir = DST / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for name in names:
        # copy images
        shutil.copy2(SRC_TRAIN_IMAGES / name, image_dir / name)

        # label txt
        txt_name = Path(name).stem + ".txt"
        with open(label_dir / txt_name, "w") as f:
            for (
                cls,
                xc,
                yc,
                w,
                h,
            ) in img_annotations[name]:
                f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

# copy test images
test_dir = DST / "images" / "test"
test_dir.mkdir(parents=True, exist_ok=True)
for img_path in sorted(SRC_TEST_IMAGES.iterdir()):
    shutil.copy2(img_path, test_dir / img_path.name)

print("완료! 디렉토리 구조:")
for p in sorted(DST.rglob("*")):
    if p.is_dir():
        count = len(list(p.iterdir()))
        print(f"  {p.relative_to(DST)}/  ({count} files)")

### YOLO를 위한 yaml 파일 생성

In [ ]:
data_yaml = {
    "path": str(DST.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(cat_map),
    "names": cat_names,  # {0: "약품명", 1: "약품명", ...}
}

yaml_path = DST / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print(f"저장: {yaml_path}")

### 학습

In [ ]:
yolo_model = YOLO("yolo11n.pt")

results = yolo_model.train(
    data=str(DST / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project=str(Path.home() / "outputs" / USERNAME / "yolo11n"),
    name="pill_yolo11n",
)


### 추론

In [ ]:
inf_yolo11_model = YOLO(
    f"runs/detect/outputs/{USERNAME}/yolo11n/pill_yolo11n/weights/best.pt"
)

results = inf_yolo11_model.predict(
    source=str(DST / "images" / "test"),
    imgsz=640,
    save=True,
    project=str(Path.home() / "outputs" / USERNAME),
    name="pill_yolo11n_predict",
)

In [ ]:
metrics = inf_yolo11_model.val(data=yaml_path)

ap_matrix = metrics.box.all_ap  # (num_classes, 10)

# IoU 0.75~0.95 = index 5~9
ap_75_95 = ap_matrix[:, 5:]  # (num_classes, 5)
map_75_95 = ap_75_95.mean()  # 전체 mAP@[0.75:0.95]
per_class = ap_75_95.mean(axis=1)  # 클래스별 mAP@[0.75:0.95]

print(f"mAP@[0.50:0.95]: {metrics.box.map:.4f}")
print(f"mAP@[0.75:0.95]: {map_75_95:.4f}")
print(f"mAP@0.50:        {metrics.box.map50:.4f}")
print(f"mAP@0.75:        {metrics.box.map75:.4f}")

---
### 정량적 평가
YOLO11n 50ep | 기본 데이터
```r
mAP@[0.50:0.95]: 0.8800
mAP@[0.75:0.95]: 0.8515
mAP@0.50:        0.9092
mAP@0.75:        0.9054
```